# Text generation using RNN - Character Level (TO BE RUN IN GOOGLE COLAB)

To generate text using RNN, we need a to convert raw text to a supervised learning problem format.

Take, for example, the following corpus:

"Her brother shook his head incredulously"

First we need to divide the data into tabular format containing input (X) and output (y) sequences. In case of a character level model, the X and y will look like this:

|      X     |  Y  |
|------------|-----|
|    Her b   |  r  |
|    er br   |  o  |
|    r bro   |  t  |
|     brot   |  h  |
|    broth   |  e  |
|    .....   |  .  |
|    .....   |  .  |
|    ulous   |  l  |
|    lousl   |  y  |

Note that in the above problem, the sequence length of X is five characters and that of y is one character. Hence, this is a many-to-one architecture. We can, however, change the number of input characters to any number of characters depending on the type of problem.

A model is trained on such data. To generate text, we simply give the model any five characters using which it predicts the next character. Then it appends the predicted character to the input sequence (on the extreme right of the sequence) and discards the first character (character on extreme left of the sequence). Then it predicts again using the new sequence and the cycle continues until a fix number of iterations. An example is shown below:

Seed text: "incre"

|      X                                            |  Y                       |
|---------------------------------------------------|--------------------------|
|                        incre                      |    < predicted char 1 >  |
|               ncre < predicted char 1 >              |    < predicted char 2 >  |
|       cre< predicted char 1 > < predicted char 2 >   |    < predicted char 3 >  |
|       re< predicted char 1 >< predicted char 2 > < predicted char 3 >   |    < predicted char 4 >  |
|                      ...                          |            ...           |

# Notebook Overview
1. Preprocess data
2. LSTM model
3. Generate code

In [1]:
!pip install gitpython

In [2]:
# import libraries
import warnings
warnings.filterwarnings("ignore")

import os
import re
import numpy as np
import random
import sys
import io
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Activation, LSTM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import get_file

# 1. Preprocess data

We're going to build a C code generator by training an RNN on a huge corpus of C code (the linux kernel code). You can download the C code used as source text from the following link:
https://github.com/torvalds/linux/tree/master/kernel

We have already downloaded the entire kernel folder and stored in a local directory

## Load C code

In [3]:
import os
import git

# Define the repository and directory path
repo_url = "https://github.com/aqwertyuiop48/upgrad_programming"
repo_path = "/content/upgrad_programming"
subdir_path = "2_Course_continuation/_2_Exam_2/4_Deep_learning/_6_Recurrent_Neural_Networks"

# Clone the repository
if not os.path.exists(repo_path):
    git.Repo.clone_from(repo_url, repo_path)

# Change the working directory to the specific folder
os.chdir(os.path.join(repo_path, subdir_path))
print("Current working directory:", os.getcwd())

Current working directory: /content/upgrad_programming/2_Course_continuation/_2_Exam_2/4_Deep_learning/_6_Recurrent_Neural_Networks


In [4]:
# set path where C files reside

print("Current working directory:", os.getcwd())

path = r"linux_kernel"

os.chdir(path)

file_names = os.listdir()
print(file_names)

Current working directory: /content/upgrad_programming/2_Course_continuation/_2_Exam_2/4_Deep_learning/_6_Recurrent_Neural_Networks
['acct.c', 'bpf', 'utsname_sysctl.c', 'kexec_elf.c', 'uid16.c', 'stackleak.c', 'audit.h', 'stacktrace.c', 'kexec_internal.h', 'kcmp.c', 'resource_kunit.c', 'auditsc.c', 'fork.c', 'cpu_pm.c', 'context_tracking.c', 'cgroup', 'kexec_file.c', 'irq_work.c', 'kthread.c', 'trace', 'taskstats.c', 'usermode_driver.c', 'dma.c', 'seccomp.c', 'hung_task.c', 'kallsyms.c', 'sys.c', 'iomem.c', 'vmcore_info.c', 'tsacct.c', 'kexec.c', 'pid_namespace.c', 'power', 'Makefile', 'gen_kheaders.sh', 'audit.c', 'stop_machine.c', 'fail_function.c', 'workqueue_internal.h', 'smp.c', 'scs.c', 'kexec_core.c', 'range.c', 'Kconfig.locks', 'user_namespace.c', 'kcov.c', 'watchdog_buddy.c', 'jump_label.c', 'ksyms_common.c', 'sysctl-test.c', 'groups.c', 'pid_sysctl.h', 'utsname.c', 'audit_fsnotify.c', 'Kconfig.hz', 'umh.c', 'panic.c', 'signal.c', 'user-return-notifier.c', 'ksysfs.c', 'Kconfi

In [5]:
# use regex to filter .c files
import re
c_names = ".*\.c$"

c_files = list()

for file in file_names:
    if re.match(c_names, file):
        c_files.append(file)

print(c_files)

['acct.c', 'utsname_sysctl.c', 'kexec_elf.c', 'uid16.c', 'stackleak.c', 'stacktrace.c', 'kcmp.c', 'resource_kunit.c', 'auditsc.c', 'fork.c', 'cpu_pm.c', 'context_tracking.c', 'kexec_file.c', 'irq_work.c', 'kthread.c', 'taskstats.c', 'usermode_driver.c', 'dma.c', 'seccomp.c', 'hung_task.c', 'kallsyms.c', 'sys.c', 'iomem.c', 'vmcore_info.c', 'tsacct.c', 'kexec.c', 'pid_namespace.c', 'audit.c', 'stop_machine.c', 'fail_function.c', 'smp.c', 'scs.c', 'kexec_core.c', 'range.c', 'user_namespace.c', 'kcov.c', 'watchdog_buddy.c', 'jump_label.c', 'ksyms_common.c', 'sysctl-test.c', 'groups.c', 'utsname.c', 'audit_fsnotify.c', 'umh.c', 'panic.c', 'signal.c', 'user-return-notifier.c', 'ksysfs.c', 'bounds.c', 'elfcorehdr.c', 'torture.c', 'softirq.c', 'padata.c', 'async.c', 'compat.c', 'audit_watch.c', 'reboot.c', 'sysctl.c', 'notifier.c', 'crash_core.c', 'tracepoint.c', 'nsproxy.c', 'resource.c', 'pid.c', 'capability.c', 'backtracetest.c', 'rseq.c', 'relay.c', 'freezer.c', 'scftorture.c', 'regset.c'

In [6]:
# load all c code in a list
full_code = list()
for file in c_files:
    code = open(file, "r", encoding='utf-8')
    full_code.append(code.read())
    code.close()

In [7]:
# let's look at how a typical C code looks like
print(full_code[20])

// SPDX-License-Identifier: GPL-2.0-only
/*
 * kallsyms.c: in-kernel printing of symbolic oopses and stack traces.
 *
 * Rewritten and vastly simplified by Rusty Russell for in-kernel
 * module loader:
 *   Copyright 2002 Rusty Russell <rusty@rustcorp.com.au> IBM Corporation
 *
 * ChangeLog:
 *
 * (25/Aug/2004) Paulo Marques <pmarques@grupopie.com>
 *      Changed the compression method from stem compression to "table lookup"
 *      compression (see scripts/kallsyms.c for a more complete description)
 */
#include <linux/kallsyms.h>
#include <linux/init.h>
#include <linux/seq_file.h>
#include <linux/fs.h>
#include <linux/kdb.h>
#include <linux/err.h>
#include <linux/proc_fs.h>
#include <linux/sched.h>	/* for cond_resched */
#include <linux/ctype.h>
#include <linux/slab.h>
#include <linux/filter.h>
#include <linux/ftrace.h>
#include <linux/kprobes.h>
#include <linux/build_bug.h>
#include <linux/compiler.h>
#include <linux/module.h>
#include <linux/kernel.h>
#include <linux/bsearch.h>
#i

In [8]:
# merge different c codes into one big c code
text = "\n".join(full_code)
print("Total number of characters in entire code: {}".format(len(text)))

Total number of characters in entire code: 2242645


In [9]:
# top_n: only consider first top_n characters and discard the rest for memory and computational efficiency
top_n = 400000
text = text[:top_n]

## Convert characters to integers

In [10]:
# create character to index mapping
chars = sorted(list(set(text)))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

In [11]:
print("Vocabulary size: {}".format(len(chars)))

Vocabulary size: 96


## Divide data in input (X) and output (y)

### Create sequences

In [12]:
# define length for each sequence
MAX_SEQ_LENGTH = 50          # number of input characters (X) in each sequence
STEP           = 3           # increment between each sequence
VOCAB_SIZE     = len(chars)  # total number of unique characters in dataset

sentences  = []              # X
next_chars = []              # y

for i in range(0, len(text) - MAX_SEQ_LENGTH, STEP):
    sentences.append(text[i: i + MAX_SEQ_LENGTH])
    next_chars.append(text[i + MAX_SEQ_LENGTH])

In [13]:
print('Number of training samples: {}'.format(len(sentences)))

Number of training samples: 133317


## Create input and output using the created sequences

When you're not using the Embedding layer of the Keras as the very first layer, you need to convert your data in the following format:
#### input shape should be of the form :  (#samples, #timesteps, #features)
#### output shape should be of the form :  (#samples, #timesteps, #features)

![Tensor shape](./jupyter resources/rnn_tensor.png)

#samples: the number of data points (or sequences)
#timesteps: It's the length of the sequence of your data (the MAX_SEQ_LENGTH variable).
#features: Number of features depends on the type of problem. In this problem, #features is the vocabulary size, that is, the dimensionality of the one-hot encoding matrix using which each character is being represented. If you're working with **images**, features size will be equal to: (height, width, channels), and the input shape will be (#training_samples, #timesteps, height, width, channels)

In [14]:
# create X and y
X = np.zeros((len(sentences), MAX_SEQ_LENGTH, VOCAB_SIZE), dtype=bool)
y = np.zeros((len(sentences), VOCAB_SIZE), dtype=bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

In [15]:
print("Shape of X: {}".format(X.shape))
print("Shape of y: {}".format(y.shape))

Shape of X: (133317, 50, 96)
Shape of y: (133317, 96)


Here, X is reshaped to (#samples, #timesteps, #features). We have explicitly mentioned the third dimension (#features) because we won't use the Embedding() layer of Keras in this case since there are only 97 characters. Characters can be represented as one-hot encoded vector. There are no word embeddings for characters.

# 2. LSTM

In [16]:
from keras.optimizers import Adam

# define model architecture - using a two-layer LSTM with 128 LSTM cells in each layer
model = Sequential()
model.add(LSTM(128, input_shape=(MAX_SEQ_LENGTH, VOCAB_SIZE), return_sequences=True, dropout=0.5))
model.add(LSTM(128, dropout=0.5))
model.add(Dense(VOCAB_SIZE, activation = "softmax"))

optimizer = Adam(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics = ['acc'])

In [17]:
# check model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 128)        │       115,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 96)             │        12,384 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 259,168 (1012.38 KB)

 Trainable params: 259,168 (1012.38 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# fit model
model.fit(X, y, batch_size=128, epochs=20)

Epoch 1/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - acc: 0.1441 - loss: 3.4095
Epoch 2/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.2414 - loss: 2.8844
Epoch 3/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.2716 - loss: 2.7512
Epoch 4/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - acc: 0.2854 - loss: 2.6733
Epoch 5/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.2983 - loss: 2.6259
Epoch 6/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3068 - loss: 2.5883
Epoch 7/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3102 - loss: 2.5639
Epoch 8/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3137 - loss: 2.5487
Epoch 9/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3203 - loss: 2.5256
Epoch 10/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3201 - loss: 2.5158
Epoch 11/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc: 0.3258 - loss: 2.4921
Epoch 12/20
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - acc:

# 3. Generate code

Create a function that will make next character predictions based on temperature. If temperature is greater than 1, the generated characters will be more versatile and diverse. On the other hand, if temperature is less than one, the generated characters will be much more conservative.

In [19]:
# define function to sample next word from a probability array based on temperature
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [20]:
np.random.multinomial(10, [0.05, 0.9, 0.05], size=2)

array([[ 2,  7,  1],
       [ 0, 10,  0]])

In [21]:
# generate code

start_index = random.randint(0, len(text) - MAX_SEQ_LENGTH - 1) # pick random code to start text generation

for diversity in [0.5, 1.0, 1.5]:
        print('-'*50, 'diversity:', diversity)

        generated = ''
        sentence = text[start_index: start_index + MAX_SEQ_LENGTH]
        generated += sentence
        print('----- Generating with seed: "' + sentence + '"')
        sys.stdout.write(generated)

        for i in range(1000):
            x_pred = np.zeros((1, MAX_SEQ_LENGTH, VOCAB_SIZE))
            for t, char in enumerate(sentence):
                x_pred[0, t, char_indices[char]] = 1.

            preds = model.predict(x_pred, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]

            generated += next_char
            sentence = sentence[1:] + next_char

            sys.stdout.write(next_char)
            sys.stdout.flush()

-------------------------------------------------- diversity: 0.5
----- Generating with seed: "(ct_idle_exit);

/**
 * ct_irq_enter - inform RCU "
(ct_idle_exit);

/**
 * ct_irq_enter - inform RCU size the anditi in whis an ete of the 'sembing shat as in potr seace fon fhe sescel ond une esice the audit_context the for if @ralted the anliting
 * @eror eutel herm the arqsen work hor ere the ard endlid on er fire ander */
	if (!atck->lonk);
	tase AUDIT_NEATE_IRE_NENEIZED | BPF_NE_ ||BABE_DESE_IN_NERNE_TEDE_DED)) |/BANIT_SSIRCEE | BPF_SEATRES_NEDIPERAN)) | BPF_SES_SER:
		return 0;
}

static void wrine_ane(struct mm_struct *tsk)
{
	struct kidit_context *context = thie;

	if (!rree>= crack_stare, lenter, &pm);
					 case_torker->ravs);
			return 0;

	return ret;

	ease AUDIT_PELA_ULE_REATEAS|ND_SEZE) | BPF_BEZE|| | BPF_SID:
		return ret;
	if (!rre = &fork_instin_&ret);
			frose_free_stark(&signal->cintly);
								                                               context->free);
		leturn ret;


In [22]:
# generate code

start_index = random.randint(0, len(text) - MAX_SEQ_LENGTH - 1) # pick random seed

for diversity in [0.5, 1.0, 1.5]:
        print('-'*50, 'diversity:', diversity)

        generated = ''
        sentence = text[start_index: start_index + MAX_SEQ_LENGTH]
        generated += sentence
        print('----- Generating with seed: "' + sentence + '"')
        sys.stdout.write(generated)

        for i in range(1000):
            x_pred = np.zeros((1, MAX_SEQ_LENGTH, VOCAB_SIZE))
            for t, char in enumerate(sentence):
                x_pred[0, t, char_indices[char]] = 1.

            preds = model.predict(x_pred, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]

            generated += next_char
            sentence = sentence[1:] + next_char

            sys.stdout.write(next_char)
            sys.stdout.flush()

-------------------------------------------------- diversity: 0.5
----- Generating with seed: "NKNOWN)
				goto out;
		} else {
			if (n->type !="
NKNOWN)
				goto out;
		} else {
			if (n->type != kthread_= ire_size,
				        context->resute
				return 0;
		work(ret,
				                                   NUGLT_SUABE_OEID_NETE_RRET_NTINE_HNDEN_EELEDENE_REREDE),
				                                 RIN_ATTADESNRSIL | &
		                                                                                                                                                                               chead_fire ce inlor be kthread_to the state thin poing when a  ind in rem.
 * @rare the anc for anc be leselore the ansigned long returns and falled when beturn in chese */

static void audit_context(struct audit_fontext *coxtext
{
	int kthread_worker *eachale_arest_caller_insert_toice unlock *
 * In mese enting in in stack the audit_context *ale the statior fall * 
 * the and af is thin size t

In [23]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-03-01 17:20:48


In [ ]:
# --- gcolab wrapper: list files in outputs/ ---
import os, datetime
_out_dir = 'outputs'
os.makedirs(_out_dir, exist_ok=True)
print(f"Files in {_out_dir}:")
_files = []
for _fn in os.listdir(_out_dir):
    _fp = os.path.join(_out_dir, _fn)
    if os.path.isfile(_fp):
        _st = os.stat(_fp)
        _files.append((datetime.datetime.fromtimestamp(_st.st_mtime).strftime('%Y-%m-%d %H:%M:%S'), _st.st_size, _fn))
_files.sort()
for _m, _s, _n in _files:
    if _s < 1024:
        _h = f"{_s} B"
    elif _s < 1024*1024:
        _h = f"{_s/1024:.2f} KB"
    else:
        _h = f"{_s/(1024*1024):.2f} MB"
    print(f"{_m}  {_h:>10}  {_n}")


In [ ]:
# --- gcolab wrapper: zip outputs/ -> outputs.zip ---
import os, shutil
os.makedirs('outputs', exist_ok=True)
shutil.make_archive('outputs', 'zip', 'outputs')
print('Created outputs.zip')
